# Modelo dimensional na Gold

O processo analisado é o registro de itens de pedidos da loja. A granularidade da fato é **uma linha por item de pedido**, identificado por `id_item`.

## Dimensões produto e cliente

Cria `dim_produto` e `dim_cliente` com chaves substitutas (`sk_produto` e `sk_cliente`) geradas por identidade. A dimensão produto incorpora a categoria, evitando outra dimensão para esse atributo.

O `MERGE` usa o identificador da origem para localizar o registro. Atualiza atributos existentes, sem manter versões históricas desses atributos (SCD tipo 1), e insere novos registros. Não remove dimensões que tenham deixado de existir na origem.

O preço utilizado na fato será o preço do item no pedido, e não o preço atual do cadastro de produtos.

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS workspace.gold.dim_produto (
    sk_produto BIGINT GENERATED BY DEFAULT AS IDENTITY,
    id_produto INT,
    nome_produto STRING,
    id_categoria INT,
    nome_categoria STRING
)
USING DELTA;

CREATE TABLE IF NOT EXISTS workspace.gold.dim_cliente (
    sk_cliente BIGINT GENERATED BY DEFAULT AS IDENTITY,
    id_cliente INT,
    nome_cliente STRING,
    email STRING
)
USING DELTA;

MERGE INTO workspace.gold.dim_produto AS destino
USING (
    SELECT
        p.id_produto,
        p.nome_produto,
        p.id_categoria,
        c.nome_categoria
    FROM workspace.silver.produtos p
    INNER JOIN workspace.silver.categorias c
        ON p.id_categoria = c.id_categoria
) AS origem
ON destino.id_produto = origem.id_produto

WHEN MATCHED THEN UPDATE SET
    destino.nome_produto = origem.nome_produto,
    destino.id_categoria = origem.id_categoria,
    destino.nome_categoria = origem.nome_categoria

WHEN NOT MATCHED THEN INSERT (
    id_produto,
    nome_produto,
    id_categoria,
    nome_categoria
)
VALUES (
    origem.id_produto,
    origem.nome_produto,
    origem.id_categoria,
    origem.nome_categoria
);

MERGE INTO workspace.gold.dim_cliente AS destino
USING workspace.silver.clientes AS origem
ON destino.id_cliente = origem.id_cliente

WHEN MATCHED THEN UPDATE SET
    destino.nome_cliente = origem.nome_cliente,
    destino.email = origem.email

WHEN NOT MATCHED THEN INSERT (
    id_cliente,
    nome_cliente,
    email
)
VALUES (
    origem.id_cliente,
    origem.nome_cliente,
    origem.email
);

SELECT 'dim_produto' AS tabela, COUNT(*) AS registros
FROM workspace.gold.dim_produto
UNION ALL
SELECT 'dim_cliente', COUNT(*)
FROM workspace.gold.dim_cliente;

## Dimensão tempo

Cria um calendário diário, do início do menor ano ao fim do maior ano presente nos pedidos da Silver. A chave `sk_tempo` usa o formato numérico `AAAAMMDD`.

Inclui data, ano, trimestre, mês, dia e dia da semana. Para os pedidos iniciais de outubro de 2026, o calendário contém os 365 dias de 2026.

In [ ]:
%sql
CREATE OR REPLACE TABLE workspace.gold.dim_tempo
USING DELTA
AS
WITH limites AS (
    SELECT
        trunc(MIN(data_pedido), 'YEAR') AS inicio,
        last_day(
            add_months(trunc(MAX(data_pedido), 'YEAR'), 11)
        ) AS fim
    FROM workspace.silver.pedidos
),
calendario AS (
    SELECT explode(
        sequence(inicio, fim, INTERVAL 1 DAY)
    ) AS data
    FROM limites
    WHERE inicio IS NOT NULL
)
SELECT
    CAST(date_format(data, 'yyyyMMdd') AS INT) AS sk_tempo,
    data,
    year(data) AS ano,
    quarter(data) AS trimestre,
    month(data) AS mes,
    dayofmonth(data) AS dia,
    dayofweek(data) AS dia_semana
FROM calendario;

## Fato de itens de pedido

Relaciona itens e pedidos da Silver com as três dimensões Gold. Mantém identificadores do item e do pedido e o status do pedido.

As medidas são quantidade, preço unitário e valor total do item (`quantidade × preco_unitario`). O preço unitário não deve ser somado como se fosse um total financeiro.

Inclui todos os status, inclusive pedidos em preparação. Portanto, o total desta tabela representa o valor dos itens de todos os pedidos, não somente vendas concluídas.

A tabela fato é reconstruída em cada carga, evitando acumular os mesmos itens em execuções repetidas.

In [ ]:
%sql
CREATE OR REPLACE TABLE workspace.gold.fato_itens_pedido
USING DELTA
AS
SELECT
    i.id_item,
    i.id_pedido,
    dp.sk_produto,
    dc.sk_cliente,
    dt.sk_tempo,
    p.status_pedido,
    i.quantidade,
    i.preco_unitario,
    CAST(
        i.quantidade * i.preco_unitario AS DECIMAL(18,2)
    ) AS valor_total
FROM workspace.silver.itens_pedido i
INNER JOIN workspace.silver.pedidos p
    ON i.id_pedido = p.id_pedido
INNER JOIN workspace.gold.dim_produto dp
    ON i.id_produto = dp.id_produto
INNER JOIN workspace.gold.dim_cliente dc
    ON p.id_cliente = dc.id_cliente
INNER JOIN workspace.gold.dim_tempo dt
    ON p.data_pedido = dt.data;

## Reconciliação da Gold

Confere se a quantidade de itens é igual à Silver, se `id_item` é único e se o total financeiro calculado na Silver coincide com o total da Gold.

Resultados iniciais: 10 produtos, 5 clientes, 365 datas, 10 itens de pedido e R$ 690,40 somando todos os status.

In [ ]:
from decimal import Decimal

fato = spark.table("workspace.gold.fato_itens_pedido")
itens = spark.table("workspace.silver.itens_pedido")

assert fato.count() == itens.count(), "Quantidade de itens diferente."

assert fato.select("id_item").distinct().count() == fato.count(), (
    "Há itens duplicados na fato."
)

total_silver = spark.sql("""
    SELECT SUM(quantidade * preco_unitario) AS total
    FROM workspace.silver.itens_pedido
""").first()["total"]

total_gold = spark.sql("""
    SELECT SUM(valor_total) AS total
    FROM workspace.gold.fato_itens_pedido
""").first()["total"]

assert total_gold == total_silver, "Total financeiro diferente."

for tabela in [
    "dim_produto",
    "dim_cliente",
    "dim_tempo",
    "fato_itens_pedido"
]:
    quantidade = spark.table(f"workspace.gold.{tabela}").count()
    print(f"{tabela}: {quantidade} registros")

print(f"Valor total dos itens de todos os pedidos: R$ {total_gold:.2f}")
print("Gold concluída e validada.")